# Day 4: Add workplace context from Work IQ

You open your inbox and find three urgent emails: the Professional Claw Hammer is out of stock at the Seattle store, customers are complaining, and colleagues are asking you to coordinate a transfer or emergency restock. You need to understand what the email thread says and find out what the company handbook says about emergency procurement escalation.

**🎯 Mission**
- Add a **Work IQ knowledge source** that can query your real Microsoft 365 data (emails, Teams, calendar)
- Build a 3-source knowledge base (HR docs + health docs + Work IQ)
- Ask a question where one sub-answer comes from your inbox and the other from company policy

## The building blocks

In Parts 1–3, your knowledge base queried documents and structured data. Now you'll add a source that connects to your personal workplace context.

- **Work IQ Knowledge Source**: connects to Microsoft 365 (Outlook, Teams, Calendar, OneDrive). The KB queries it at runtime to surface emails, chats, and calendar events relevant to the user's question.
- **Delegated token**: Work IQ accesses data on behalf of the signed-in user. This notebook uses `AzureCliCredential` to obtain a delegated token from your Azure CLI sign-in.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## Step 2: Login to your Azure account

This notebook uses `AzureCliCredential` to fetch a delegated query token for Work IQ, so you must sign in to the Azure CLI before running the Work IQ retrieval. If you already did this for the previous Fabric IQ notebook, you don't need to login again.

Run the cell below to generate a login command, then run that command in the VS Code Terminal:

In [ ]:
print(f"az login --use-device-code --tenant \"{AZURE_TENANT_ID}\"")

## Step 3: Acquire your delegated user token

In order for the knowledge base to make queries to a Work IQ knowledge source, it requires a delegated user token.

In this step, you will acquire a delegated user token for the Azure AI Search scope, so that the knowledge base is allowed to access Work IQ on your behalf during retrieval later.

This code uses `AzureCliCredential`, so the token comes from the Azure CLI account that you logged in with.


In [ ]:
import base64
from azure.identity import AzureCliCredential

user_credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID, process_timeout=60)
user_token = user_credential.get_token("https://search.azure.com/.default").token

# Decode JWT payload to verify the signed-in account
payload_b64 = user_token.split(".")[1]
payload_b64 += "=" * (-len(payload_b64) % 4)
user_token_claims = json.loads(base64.urlsafe_b64decode(payload_b64))

unique_name = user_token_claims.get("unique_name")
if unique_name:
    print("Acquired token for Azure AI Search query source")
    print(f"Signed in as: {unique_name}")
else:
    print("❌ No unique_name found in the token. The CLI may be signed in to the wrong account.")
    print(f"   Run: az login --use-device-code --tenant {AZURE_TENANT_ID}")
    print("   Then re-run this cell.")


## Step 4: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Work IQ knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `workiq-knowledge-source`: Points to Work IQ for workplace context

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


## Step 5: Create the multi-source + Work IQ knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context (emails, chats, events, meetings, etc). Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 6: Query the knowledge base

Ask a two-part question: one half needs your actual inbox context from Work IQ, the other needs company policy from the HR index.

* _"What are my recent emails about the Professional Claw Hammer stock issue at Seattle?"_ → should come from **Work IQ**
* _"What does the Zava handbook say about emergency procurement escalation?"_ → should come from `hrdocs`

> ⏳ Expect to wait 1-2 minutes for the response, as Work IQ increases retrieval latency. The retrieval request specifies `max_runtime_in_seconds` to increase the time the knowledge base waits for the results from the sources.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("Check my recent work emails for messages about the Professional Claw Hammer. "
            "Summarize what colleagues are saying and what actions have been requested. " 
            "Which Zava job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


### Review the references

The next cell prints the raw `references` array returned by the knowledge base so you can inspect which sources contributed to the answer. For the Work IQ knowledge source, the references have `type: "workIQ"`. The `sourceData` contains `extracts` with a synthesized answer from the Work IQ agent, and each reference contains `attributions` with URLs that link to any referenced emails, documents, etc.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which knowledge source answered the **email thread summary** question?
2. Which knowledge source answered the **job roles** question?

## ✅ Mission complete

**What you built:**

* ✓ **Work IQ Knowledge Source**: A source that connects your real Microsoft 365 data to the knowledge base, so the KB can retrieve emails, Teams messages, and calendar context at query time.
* ✓ **3-source Knowledge Base**: A single KB spanning HR docs, health docs, and live M365 workplace data, with the agent routing each sub-question to the right source.
* ✓ **Hybrid inbox and policy retrieval**: Work IQ surfaced the email thread about the Seattle stock outage. The HR index answered the budget management question.

➡️ Continue to: [Part 5: Combine Work IQ and Fabric IQ](part5-work-iq-fabric-iq-to-kb.ipynb).